# REGRESSÃO LOGÍSTICA BINÁRIA

A Regressão Logística é uma técnica estatística usada para modelar a probabilidade de um evento ocorrer,
baseada em uma combinação de variáveis independentes. Diferente da regressão linear, que prevê valores numéricos,
a regressão logística é usada para prever resultados categóricos (sim/não, 0/1).

Aqui está a explicação em 3 pontos simples:

   1. O Objetivo: Modelar a probabilidade de um evento (ex: "será que o pacote chegará atrasado?")
      com base em variáveis que já temos (ex: distância, número de semáforos).
   2. A "Função Logística": O computador ajusta uma curva em S (sigmoide) que
      separa a "probabilidade de sucesso" da "probabilidade de falha".
   3. A Previsão: Com essa curva pronta, se você me der novos valores,
      o modelo me diz a probabilidade (ex: "70% de chance de atraso").

   A fórmula básica que você verá no notebook é:
   P(y=1) = 1 / (1 + e^-(α + βX))
    * P: Probabilidade do evento ocorrer.
    * e: Número de Euler (~2.718).
    * α e β: São os números que o modelo aprende para ajustar a curva.

   Em resumo: é transformar dados em uma curva em S para prever probabilidades de eventos com base no passado.
</markdown_cell>


#### EXEMPLO 1 - Previsão de atraso no envio de pacotes


1 - Importação dos pacotes
</markdown_cell>


In [1]:
import pandas as pd # manipulação de dados em formato de dataframe
import numpy as np # operações matemáticas
import seaborn as sns # visualização gráfica
import matplotlib.pyplot as plt # visualização gráfica
from scipy.interpolate import UnivariateSpline # curva sigmoide suavizada
import statsmodels.api as sm # estimação de modelos
from statstests.process import stepwise # procedimento Stepwise
from scipy import stats # estatística chi2
import plotly.graph_objects as go # gráficos 3D
from statsmodels.iolib.summary2 import summary_col # comparação entre modelos
from statsmodels.discrete.discrete_model import MNLogit # estimação do modelo
                                                        #logístico multinomial

import warnings
warnings.filterwarnings("ignore")

2 - Carregando Dados
</markdown_cell>


In [2]:
# CARREGAMENTO DA BASE DE DADOS
df_atrasado = pd.read_csv('atrasado.csv', delimiter=',')
df_atrasado

FileNotFoundError: [Errno 2] No such file or directory: 'atrasado.csv'

2.1 - Características das variáveis do dataset
</markdown_cell>


In [ ]:
# Características das variáveis do dataset
df_atrasado.info()

2.2 - Estatísticas univariadas
</markdown_cell>


In [ ]:
# Estatísticas univariadas
df_atrasado.describe()

2.3 - Frequência da variável dependente
</markdown_cell>


In [ ]:
# Tabela de frequências absolutas da variável 'atrasado'
df_atrasado['atrasado'].value_counts().sort_index()

3 - Gráfico de dispersão com o ajuste logístico
</markdown_cell>


In [ ]:
# Ajuste logístico entre a variável dependente e a variável 'sem'

plt.figure(figsize=(15,10))
sns.regplot(x=df_atrasado["sem"], y=df_atrasado["atrasado"],
            ci=None, marker="o", logistic=True,
            scatter_kws={"color":"orange", "s":250, "alpha":0.7},
            line_kws={"color":"darkorchid", "linewidth":7})
plt.axhline(y=0.5, color="grey", linestyle=":")
plt.xlabel("Quantidade de Semáforos", fontsize=20)
plt.ylabel("Atrasado", fontsize=20)
plt.xticks(np.arange(0, df_atrasado["sem"].max() + 0.01), fontsize=14)
plt.yticks(np.arange(0, 1.1, 0.2), fontsize=14)
plt.show()

4 - Estimação do modelo logístico binário
</markdown_cell>


In [ ]:
# Estimação do modelo
modelo_atrasos = sm.Logit.from_formula('atrasado ~ dist + sem', df_atrasado).fit()

4.1 - Observação dos parâmetros resultantes da estimação
</markdown_cell>


In [ ]:
print(modelo_atrasos.summary())

4.2 - Comparação entre modelos usando summary_col
</markdown_cell>


In [ ]:
summary_col([modelo_atrasos],
            model_names=["MODELO"],
            stars=True,
            info_dict = {
                "N":lambda x: "{0:d}".format(int(x.nobs)),
                "Log-lik":lambda x: "{:.3f}".format(x.llf)
        })

# Análise da Regressão Logística Binária (Logistic Regression)

## Resumo do Modelo

O objetivo deste modelo é analisar a relação entre as variáveis explicativas
("distância" e "número de semáforos") e a variável resposta binária "atrasado".

### Equação Logística

[
P(atrasado=1) = 1 / (1 + e^-(α + β₁×dist + β₂×sem))
]

Onde:

* **Intercepto (α):** Valor base quando distância e semáforos são zero.
* **Coeficiente da Distância (β₁):** Impacto da distância na probabilidade de atraso.
* **Coeficiente dos Semáforos (β₂):** Impacto do número de semáforos na probabilidade de atraso.

---

# Interpretação dos Coeficientes

Os coeficientes da regressão logística estão em log-odds (logit). Para interpretar como probabilidades,
precisamos converter para odds usando e^β e depois para probabilidades.

---

# Qualidade do Ajuste

## Log-Likelihood

O log-likelihood mede a probabilidade dos dados dado o modelo. Valores mais próximos de zero indicam melhor ajuste.

## Pseudo R² (McFadden)

Diferente do R² da regressão linear, o pseudo R² da regressão logística mede a melhora do modelo em relação ao modelo nulo (sem variáveis).

---

# Significância do Modelo

## Teste Likelihood Ratio

O teste LR compara o modelo com e sem as variáveis explicativas. Um p-valor baixo indica que as variáveis contribuem significativamente para o modelo.

---

# Análise dos Coeficientes Individualmente

## Intercepto

| Coeficiente | Erro Padrão | z | p-valor |
| ----------- | ----------- | - | ------- |

### Interpretação

O intercepto representa o log-odds de atraso quando distância e semáforos são zero.

---

## Distância

| Coeficiente | Erro Padrão | z | p-valor |
| ----------- | ----------- | - | ------- |

### Interpretação

O coeficiente da distância indica como cada unidade adicional de distância afeta a probabilidade de atraso.

---

## Semáforos

| Coeficiente | Erro Padrão | z | p-valor |
| ----------- | ----------- | - | ------- |

### Interpretação

O coeficiente dos semáforos indica como cada semáforo adicional afeta a probabilidade de atraso.

---

## Odds Ratios (Razões de Probabilidade)

Para facilitar a interpretação, podemos exponenciar os coeficientes:

[
odds_ratio = e^β
]

Uma odds-ratio > 1 indica que o fator aumenta a probabilidade de atraso,
enquanto odds-ratio < 1 indica que diminui a probabilidade.

---

# Previsões com o Modelo

## Probabilidades previstas para o dataset


In [ ]:
df_atrasado['phat'] = modelo_atrasos.predict()
df_atrasado
</code_cell>


## Exemplo de previsão individual

Qual a probabilidade de atraso para um trajeto de 7 km com 10 semáforos?


In [ ]:
modelo_atrasos.predict(pd.DataFrame({'dist':[7], 'sem':[10]}))

---

# Curva ROC e Métricas de Classificação

## Matriz de Confusão


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay, recall_score

def matriz_confusao(predicts, observado, cutoff):
    values = predicts.values
    predicao_binaria = []
    for item in values:
        if item < cutoff:
            predicao_binaria.append(0)
        else:
            predicao_binaria.append(1)
    cm = confusion_matrix(predicao_binaria, observado)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.xlabel("True")
    plt.ylabel("Classified")
    plt.gca().invert_xaxis()
    plt.gca().invert_yaxis()
    plt.show()
    sensitividade = recall_score(observado, predicao_binaria, pos_label=1)
    especificidade = recall_score(observado, predicao_binaria, pos_label=0)
    acuracia = accuracy_score(observado, predicao_binaria)
    indicadores = pd.DataFrame({"Sensitividade":[sensitividade],
                                "Especificidade":[especificidade],
                                "Acurácia":[acuracia]})
    return indicadores

In [ ]:
matriz_confusao(observado=df_atrasado["atrasado"], predicts=df_atrasado["phat"], cutoff=0.5)

## Igualando critérios de especificidade e de sensitividade


In [ ]:
def espec_sens(observado, predicts):
    values = predicts.values
    cutoffs = np.arange(0, 1.01, 0.01)
    lista_sensitividade = []
    lista_especificidade = []
    for cutoff in cutoffs:
        predicao_binaria = []
        for item in values:
            if item >= cutoff:
                predicao_binaria.append(1)
            else:
                predicao_binaria.append(0)
        sensitividade = recall_score(observado, predicao_binaria, pos_label=1)
        especificidade = recall_score(observado, predicao_binaria, pos_label=0)
        lista_sensitividade.append(sensitividade)
        lista_especificidade.append(especificidade)
    resultado = pd.DataFrame({"cutoffs":cutoffs,"sensitividade":lista_sensitividade,"especificidade":lista_especificidade})
    return resultado

dados_plotagem = espec_sens(observado=df_atrasado["atrasado"], predicts=df_atrasado["phat"])
dados_plotagem

In [ ]:
plt.figure(figsize=(15,10))
with plt.style.context("seaborn-v0_8-whitegrid"):
    plt.plot(dados_plotagem.cutoffs, dados_plotagem.sensitividade, marker="o", color="indigo", markersize=8)
    plt.plot(dados_plotagem.cutoffs, dados_plotagem.especificidade, marker="o", color="darkorange", markersize=8)
plt.xlabel("Cutoff", fontsize=20)
plt.ylabel("Sensitividade / Especificidade", fontsize=20)
plt.xticks(np.arange(0, 1.1, 0.2), fontsize=14)
plt.yticks(np.arange(0, 1.1, 0.2), fontsize=14)
plt.legend(["Sensitividade", "Especificidade"], fontsize=20)
plt.show()

## Construção da curva ROC


In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(df_atrasado["atrasado"], df_atrasado["phat"])
roc_auc = auc(fpr, tpr)
gini = (roc_auc - 0.5) / 0.5

plt.figure(figsize=(15,10))
plt.plot(fpr, tpr, marker="o", color="darkorchid", markersize=11, linewidth=3)
plt.plot(fpr, fpr, color="gray", linestyle="dashed")
plt.title("Área abaixo da curva: %g" % round(roc_auc, 4) + " | Coeficiente de GINI: %g" % round(gini, 4), fontsize=22)
plt.xlabel("1 - Especificidade", fontsize=20)
plt.ylabel("Sensitividade", fontsize=20)
plt.xticks(np.arange(0, 1.1, 0.2), fontsize=14)
plt.yticks(np.arange(0, 1.1, 0.2), fontsize=14)
plt.show()

# Resumo: Regressão Logística Binária

Este documento apresenta um resumo do fluxo de código do notebook, focado na análise da relação entre
**Distância**, **Semáforos** e **Atraso**.

---

## 1. Preparação e Exploração
- **Objetivo:** Entender como a "Distância" e o número de "Semáforos" influenciam a probabilidade de atraso.
- **Dados:** O arquivo `atrasado.csv` é carregado.
- **Análise Inicial:** Uso de `describe()` para estatísticas descritivas e `info()` para verificar tipos de dados.

## 2. Visualização Inicial
- É gerado um gráfico de dispersão com uma curva logística usando `seaborn.regplot(logistic=True)`.
- Isso permite visualizar se os pontos seguem uma tendência não linear (curva em S).

## 3. Construção do Modelo
O modelo de **Logística Binária** é estimado:
```python
modelo_atrasos = sm.Logit.from_formula("atrasado ~ dist + sem", df_atrasado).fit()
```
- A fórmula `"atrasado ~ dist + sem"` indica que o atraso depende da distância e dos semáforos.
- O `fit()` faz o computador encontrar os melhores números para a curva logística.

## 4. Interpretação dos Resultados
Ao rodar `modelo_atrasos.summary()`, obtemos:
- **Coeficientes:** Indicam como cada variável afeta o log-odds de atraso.
- **P-value:** Como é menor que 0,05, a variável é estatisticamente significativa para prever o atraso.

    **Na ciência, definimos uma "linha de corte" chamada Nível de Significância ($\alpha$). O padrão é 0,05 (5%).**

## 5. Previsões e Métricas
O código salva os resultados individuais no DataFrame:
- **phat:** A probabilidade prevista de atraso para cada observação.
- **Matriz de confusão:** Mede os acertos e erros do modelo com cutoff de 0,5.
- **ROC / AUC:** Mede a capacidade do modelo de separar atrasados de não atrasados.

---
**Conclusão do Notebook:** O modelo logístico é utilizado para prever a probabilidade de atraso com base em distância e número de semáforos.
</markdown_cell>